In [ ]:
!pip install scikit-learn pandas numpy nltk matplotlib seaborn mysql-connector-python flask ngrok

# For Google Colab only
!pip install pyngrok

In [ ]:
import pandas as pd
import numpy as np
import nltk
import re
import string
import mysql.connector
from mysql.connector import Error
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
import warnings
warnings.filterwarnings('ignore')

# Download NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

In [ ]:
# Complete Fake News Detection System - No MySQL Required
# Works perfectly in Google Colab and Jupyter Notebook

import pandas as pd
import numpy as np
import nltk
import re
import string
import json
import os
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
import warnings
warnings.filterwarnings('ignore')

# Download NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("✅ All libraries imported successfully!")

In [ ]:
# Data Storage Module (Replaces MySQL)
class DataStorage:
    def __init__(self, storage_file='prediction_history.csv'):
        self.storage_file = storage_file
        self.initialize_storage()

    def initialize_storage(self):
        """Create storage file if it doesn't exist"""
        if not os.path.exists(self.storage_file):
            df = pd.DataFrame(columns=['timestamp', 'news_text', 'predicted_label',
                                       'confidence', 'model_used', 'prediction_time'])
            df.to_csv(self.storage_file, index=False)

    def save_prediction(self, news_text, predicted_label, confidence, model_used):
        """Save prediction to CSV file"""
        new_record = pd.DataFrame([{
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'news_text': news_text[:500],  # Store first 500 chars
            'predicted_label': predicted_label,
            'confidence': confidence,
            'model_used': model_used,
            'prediction_time': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }])

        # Append to existing file
        if os.path.exists(self.storage_file):
            existing = pd.read_csv(self.storage_file)
            updated = pd.concat([existing, new_record], ignore_index=True)
        else:
            updated = new_record

        updated.to_csv(self.storage_file, index=False)
        return True

    def get_statistics(self):
        """Get statistics from stored predictions"""
        if os.path.exists(self.storage_file):
            df = pd.read_csv(self.storage_file)
            if len(df) > 0:
                stats = {
                    'total_predictions': len(df),
                    'real_count': len(df[df['predicted_label'] == 1]),
                    'fake_count': len(df[df['predicted_label'] == 0]),
                    'avg_confidence': df['confidence'].mean() if 'confidence' in df else 0
                }
                return stats
        return {
            'total_predictions': 0,
            'real_count': 0,
            'fake_count': 0,
            'avg_confidence': 0
        }

    def get_history(self, limit=20):
        """Get prediction history"""
        if os.path.exists(self.storage_file):
            df = pd.read_csv(self.storage_file)
            return df.tail(limit)
        return pd.DataFrame()

print("✅ Data Storage Module Ready (CSV-based, no MySQL needed)")

In [ ]:
# Data Preprocessor Module
class DataPreprocessor:
    def __init__(self):
        self.stemmer = PorterStemmer()
        self.stop_words = set(stopwords.words('english'))

    def clean_text(self, text):
        """Clean and preprocess text"""
        if pd.isna(text):
            return ""

        # Convert to lowercase
        text = str(text).lower()

        # Remove punctuation
        text = text.translate(str.maketrans('', '', string.punctuation))

        # Remove numbers
        text = re.sub(r'\d+', '', text)

        # Remove extra whitespace
        text = re.sub(r'\s+', ' ', text).strip()

        return text

    def tokenize_and_stem(self, text):
        """Tokenize and apply stemming"""
        try:
            tokens = word_tokenize(text)
            # Remove stopwords and stem
            tokens = [self.stemmer.stem(token) for token in tokens
                     if token not in self.stop_words and len(token) > 2]
            return ' '.join(tokens)
        except:
            return text

    def preprocess_dataset(self, df):
        """Full preprocessing pipeline"""
        # Handle missing values
        if 'text' in df.columns:
            df['text'] = df['text'].fillna('')
        if 'title' in df.columns:
            df['title'] = df['title'].fillna('')
            df['combined_text'] = df['title'] + ' ' + df['text']
        else:
            df['combined_text'] = df['text']

        # Clean text
        df['cleaned_text'] = df['combined_text'].apply(self.clean_text)

        # Tokenize and stem
        df['processed_text'] = df['cleaned_text'].apply(self.tokenize_and_stem)

        return df

print("✅ Data Preprocessor Module Ready")

In [ ]:
# Fake News Detector Module
class FakeNewsDetector:
    def __init__(self):
        self.vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
        self.models = {
            'Naive Bayes': MultinomialNB(),
            'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
            'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
        }
        self.trained_models = {}
        self.best_model = None
        self.best_model_name = None

    def extract_features(self, texts):
        """Convert text to TF-IDF features"""
        return self.vectorizer.fit_transform(texts)

    def transform_features(self, texts):
        """Transform new texts using fitted vectorizer"""
        return self.vectorizer.transform(texts)

    def train_models(self, X_train, y_train):
        """Train multiple models"""
        for name, model in self.models.items():
            print(f"Training {name}...")
            model.fit(X_train, y_train)
            self.trained_models[name] = model
        return self.trained_models

    def evaluate_models(self, X_test, y_test):
        """Evaluate all trained models"""
        evaluation_results = {}

        for name, model in self.trained_models.items():
            y_pred = model.predict(X_test)

            evaluation_results[name] = {
                'Accuracy': accuracy_score(y_test, y_pred),
                'Precision': precision_score(y_test, y_pred, zero_division=0),
                'Recall': recall_score(y_test, y_pred, zero_division=0),
                'F1 Score': f1_score(y_test, y_pred, zero_division=0)
            }

        # Select best model (based on F1 score)
        best_f1 = 0
        for name, metrics in evaluation_results.items():
            if metrics['F1 Score'] > best_f1:
                best_f1 = metrics['F1 Score']
                self.best_model = self.trained_models[name]
                self.best_model_name = name

        return evaluation_results

    def predict_news(self, text, preprocessor):
        """Predict single news article"""
        # Preprocess text
        cleaned = preprocessor.clean_text(text)
        processed = preprocessor.tokenize_and_stem(cleaned)

        # Extract features
        features = self.transform_features([processed])

        # Predict
        prediction = self.best_model.predict(features)[0]

        # Get confidence (probability)
        try:
            if hasattr(self.best_model, 'predict_proba'):
                confidence = self.best_model.predict_proba(features)[0].max()
            else:
                confidence = 0.5
        except:
            confidence = 0.5

        return prediction, confidence

print("✅ Fake News Detector Module Ready")

In [ ]:
# Visualization Module
class VisualizationModule:
    @staticmethod
    def plot_model_comparison(results):
        """Plot model performance comparison"""
        models = list(results.keys())
        metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']

        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        axes = axes.ravel()

        for idx, metric in enumerate(metrics):
            values = [results[model][metric] for model in models]
            colors = ['#2ecc71' if v == max(values) else '#e74c3c' for v in values]

            bars = axes[idx].bar(models, values, color=colors)
            axes[idx].set_title(f'{metric} Comparison', fontsize=14, fontweight='bold')
            axes[idx].set_ylabel(metric)
            axes[idx].set_ylim(0, 1)
            axes[idx].tick_params(axis='x', rotation=45)

            # Add value labels on bars
            for bar, value in zip(bars, values):
                axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                             f'{value:.3f}', ha='center', va='bottom', fontweight='bold')

        plt.tight_layout()
        plt.show()

    @staticmethod
    def plot_confusion_matrix(y_true, y_pred):
        """Plot confusion matrix"""
        cm = confusion_matrix(y_true, y_pred)

        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=['Fake', 'Real'],
                    yticklabels=['Fake', 'Real'])
        plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
        plt.xlabel('Predicted Label', fontsize=12)
        plt.ylabel('Actual Label', fontsize=12)
        plt.show()

    @staticmethod
    def plot_performance_metrics(metrics_dict):
        """Plot performance metrics as a bar chart"""
        plt.figure(figsize=(10, 6))
        metrics_names = list(metrics_dict.keys())
        metrics_values = list(metrics_dict.values())

        colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']
        bars = plt.bar(metrics_names, metrics_values, color=colors, edgecolor='black')

        plt.title('Model Performance Metrics', fontsize=16, fontweight='bold')
        plt.ylabel('Score', fontsize=12)
        plt.ylim(0, 1)
        plt.grid(axis='y', alpha=0.3)

        # Add value labels
        for bar, value in zip(bars, metrics_values):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{value:.3f}', ha='center', va='bottom', fontweight='bold')

        plt.tight_layout()
        plt.show()

print("✅ Visualization Module Ready")

In [ ]:
# Interactive UI Module
from ipywidgets import widgets, Layout
from IPython.display import display, clear_output, HTML

class FakeNewsUI:
    def __init__(self, detector, preprocessor, storage):
        self.detector = detector
        self.preprocessor = preprocessor
        self.storage = storage

    def create_ui(self):
        """Create interactive UI"""

        # Title Section
        display(HTML("""
            <div style="text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                        border-radius: 15px; margin-bottom: 20px;">
                <h1 style="color: white; margin: 0;">📰 Fake News Detection System</h1>
                <p style="color: white; margin-top: 10px;">Powered by Machine Learning & NLP | <strong>No MySQL Required</strong></p>
            </div>
        """))

        # Create tabs
        tab = widgets.Tab()
        tab.children = [
            self.create_prediction_tab(),
            self.create_dashboard_tab(),
            self.create_history_tab()
        ]
        tab.set_title(0, '🔍 Predict News')
        tab.set_title(1, '📊 Dashboard')
        tab.set_title(2, '📜 History')

        display(tab)

    def create_prediction_tab(self):
        """Create prediction tab"""
        box = widgets.VBox()

        # News input area
        news_label = widgets.HTML("<h3>📝 Enter News Article Text:</h3>")

        self.news_input = widgets.Textarea(
            placeholder='Enter news article text here to analyze...\n\nExample:\n"Scientists at MIT have developed a new technology that can detect fake news with 99% accuracy. The system uses advanced AI algorithms to analyze text patterns and identify misinformation."',
            layout=Layout(width='100%', height='200px'),
            style={'description_width': 'initial'}
        )

        # Example news buttons
        example_label = widgets.HTML("<b>📌 Try Examples:</b>")

        real_example = widgets.Button(
            description='📰 Real News Example',
            button_style='success',
            layout=Layout(width='auto', margin='5px')
        )

        fake_example = widgets.Button(
            description='⚠️ Fake News Example',
            button_style='danger',
            layout=Layout(width='auto', margin='5px')
        )

        real_example.on_click(self.load_real_example)
        fake_example.on_click(self.load_fake_example)

        # Predict button
        self.predict_btn = widgets.Button(
            description='🔍 Detect Fake News',
            button_style='primary',
            layout=Layout(width='100%', margin='10px 0', padding='10px')
        )
        self.predict_btn.on_click(self.predict_news)

        # Clear button
        self.clear_btn = widgets.Button(
            description='🗑️ Clear Text',
            button_style='warning',
            layout=Layout(width='100%', margin='5px 0')
        )
        self.clear_btn.on_click(self.clear_input)

        # Output area
        self.output = widgets.Output(layout=Layout(margin='20px 0'))

        # Arrange widgets
        box.children = [
            news_label,
            self.news_input,
            example_label,
            widgets.HBox([real_example, fake_example]),
            self.predict_btn,
            self.clear_btn,
            self.output
        ]

        return box

    def load_real_example(self, b):
        self.news_input.value = """Breaking: Scientists at Stanford University have discovered a new method to recycle plastic waste using bacteria. The research, published in Science journal, shows that the engineered bacteria can break down PET plastic in days instead of centuries. This breakthrough could help solve the global plastic pollution crisis."""

    def load_fake_example(self, b):
        self.news_input.value = """SHOCKING: The government has secretly admitted that aliens exist and have been living among us for decades. According to anonymous sources inside the White House, a secret agreement was signed with extraterrestrials in 1947. The mainstream media is hiding this truth from the public!"""

    def predict_news(self, b):
        with self.output:
            clear_output()

            news_text = self.news_input.value.strip()

            if not news_text:
                display(HTML("<div style='color: red; padding: 15px; background: #fee; border-radius: 10px;'>❌ Please enter some news text to analyze!</div>"))
                return

            # Show loading
            display(HTML("<div style='text-align: center; padding: 20px;'>🔄 Analyzing news article with ML models...</div>"))

            # Make prediction
            prediction, confidence = self.detector.predict_news(news_text, self.preprocessor)

            # Save to storage
            self.storage.save_prediction(news_text, prediction, confidence, self.detector.best_model_name)

            # Display result
            if prediction == 1:
                display(HTML(f"""
                    <div style="background: linear-gradient(135deg, #d4edda 0%, #c3e6cb 100%);
                                border: 3px solid #28a745; border-radius: 15px; padding: 25px;
                                margin-top: 20px; text-align: center;">
                        <h1 style="color: #155724; margin: 0;">✅ REAL NEWS</h1>
                        <h2 style="color: #155724;">📰 This news appears to be legitimate</h2>
                        <hr>
                        <p><strong>Confidence Score:</strong> {confidence*100:.2f}%</p>
                        <p><strong>Model Used:</strong> {self.detector.best_model_name}</p>
                        <p><strong>Analysis Time:</strong> {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
                    </div>
                """))
            else:
                display(HTML(f"""
                    <div style="background: linear-gradient(135deg, #f8d7da 0%, #f5c6cb 100%);
                                border: 3px solid #dc3545; border-radius: 15px; padding: 25px;
                                margin-top: 20px; text-align: center;">
                        <h1 style="color: #721c24; margin: 0;">⚠️ FAKE NEWS</h1>
                        <h2 style="color: #721c24;">🚫 This news appears to be misinformation</h2>
                        <hr>
                        <p><strong>Confidence Score:</strong> {confidence*100:.2f}%</p>
                        <p><strong>Model Used:</strong> {self.detector.best_model_name}</p>
                        <p><strong>Analysis Time:</strong> {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
                    </div>
                """))

            # Show analyzed text preview
            display(HTML(f"""
                <div style="background: #f8f9fa; border-radius: 10px; padding: 15px; margin-top: 15px;">
                    <p><strong>📄 Analyzed Text Preview:</strong></p>
                    <p style="color: #666; font-style: italic;">"{news_text[:300]}{'...' if len(news_text) > 300 else ''}"</p>
                </div>
            """))

    def clear_input(self, b):
        self.news_input.value = ""
        with self.output:
            clear_output()

    def create_dashboard_tab(self):
        """Create dashboard tab"""
        box = widgets.VBox()
        self.dashboard_output = widgets.Output()

        refresh_btn = widgets.Button(description='🔄 Refresh Dashboard', button_style='info')
        refresh_btn.on_click(self.refresh_dashboard)

        box.children = [refresh_btn, self.dashboard_output]

        # Initial dashboard load
        self.refresh_dashboard(None)

        return box

    def refresh_dashboard(self, b):
        with self.dashboard_output:
            clear_output()
            self.display_dashboard()

    def display_dashboard(self):
        """Display analytics dashboard"""
        stats = self.storage.get_statistics()

        if stats['total_predictions'] > 0:
            # Metrics cards
            display(HTML(f"""
                <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 15px; margin: 20px 0;">
                    <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                                padding: 20px; border-radius: 10px; text-align: center; color: white;">
                        <h3>📊 Total</h3>
                        <h2>{stats['total_predictions']}</h2>
                        <p>Predictions</p>
                    </div>
                    <div style="background: linear-gradient(135deg, #27ae60 0%, #2ecc71 100%);
                                padding: 20px; border-radius: 10px; text-align: center; color: white;">
                        <h3>📰 Real</h3>
                        <h2>{stats['real_count']}</h2>
                        <p>{stats['real_count']/stats['total_predictions']*100:.1f}%</p>
                    </div>
                    <div style="background: linear-gradient(135deg, #e74c3c 0%, #c0392b 100%);
                                padding: 20px; border-radius: 10px; text-align: center; color: white;">
                        <h3>🚫 Fake</h3>
                        <h2>{stats['fake_count']}</h2>
                        <p>{stats['fake_count']/stats['total_predictions']*100:.1f}%</p>
                    </div>
                    <div style="background: linear-gradient(135deg, #f39c12 0%, #e67e22 100%);
                                padding: 20px; border-radius: 10px; text-align: center; color: white;">
                        <h3>🎯 Avg Conf.</h3>
                        <h2>{stats['avg_confidence']*100:.1f}%</h2>
                        <p>Confidence</p>
                    </div>
                </div>
            """))

            # Create pie chart
            plt.figure(figsize=(8, 6))
            labels = ['Real News', 'Fake News']
            sizes = [stats['real_count'], stats['fake_count']]
            colors = ['#27ae60', '#e74c3c']
            explode = (0.05, 0.05)

            plt.pie(sizes, explode=explode, labels=labels, colors=colors,
                    autopct='%1.1f%%', shadow=True, startangle=90)
            plt.title('Prediction Distribution', fontsize=16, fontweight='bold')
            plt.show()

        else:
            display(HTML("""
                <div style="text-align: center; padding: 50px; background: #f8f9fa; border-radius: 10px;">
                    <h3>📊 No predictions yet!</h3>
                    <p>Go to the Predict News tab to start analyzing articles.</p>
                </div>
            """))

    def create_history_tab(self):
        """Create history tab"""
        box = widgets.VBox()
        self.history_output = widgets.Output()

        refresh_btn = widgets.Button(description='🔄 Refresh History', button_style='info')
        refresh_btn.on_click(self.refresh_history)

        box.children = [refresh_btn, self.history_output]

        self.refresh_history(None)

        return box

    def refresh_history(self, b):
        with self.history_output:
            clear_output()
            history_df = self.storage.get_history()

            if len(history_df) > 0:
                display_df = history_df[['timestamp', 'news_text', 'predicted_label', 'confidence']].copy()
                display_df['predicted_label'] = display_df['predicted_label'].map({1: '✅ REAL', 0: '⚠️ FAKE'})
                display_df['confidence'] = display_df['confidence'].apply(lambda x: f'{x*100:.1f}%')
                display_df.columns = ['Time', 'News Preview', 'Prediction', 'Confidence']

                print(f"📜 Last {len(display_df)} Predictions:")
                display(display_df)
            else:
                display(HTML("""
                    <div style="text-align: center; padding: 50px; background: #f8f9fa; border-radius: 10px;">
                        <h3>📜 No prediction history yet!</h3>
                        <p>Start analyzing news articles to see history here.</p>
                    </div>
                """))

print("✅ UI Module Ready")

In [ ]:
# Load Dataset and Train Models
def create_training_dataset():
    """Create a comprehensive training dataset"""

    training_data = {
        'text': [
            # Real news examples (label=1)
            "Scientists discover new treatment for cancer patients at research hospital",
            "Government announces new policies to improve education system nationwide",
            "Company reports quarterly earnings exceeding market expectations",
            "Researchers publish peer-reviewed study on climate change effects",
            "President signs bipartisan infrastructure bill into law today",
            "Local community raises funds for new school playground equipment",
            "Olympic athlete breaks world record in百米 final competition",
            "Technology company launches innovative product for renewable energy",
            "Health officials recommend annual flu vaccination for all citizens",
            "Stock market reaches record high following positive economic data",
            "University researchers develop breakthrough in battery technology",
            "International summit reaches agreement on trade relations",
            "New study shows benefits of meditation for mental health",
            "Space agency successfully launches satellite into orbit",
            "Farmers report record harvest due to favorable weather conditions",

            # Fake news examples (label=0)
            "SHOCKING: Celebrity death hoax goes viral on social media platforms",
            "Miracle cure company claims product can heal all diseases instantly",
            "BREAKING: World leaders secretly admit to alien contact conspiracy",
            "Fake news spreads faster than truth according to misleading study",
            "Government hiding evidence of supernatural phenomenon in area",
            "Famous politician caught in fabricated scandal by unreliable sources",
            "Clickbait headline exaggerates scientific findings beyond recognition",
            "Satirical website article mistaken for real news by readers",
            "Misleading advertisement claims weight loss without any effort",
            "Fabricated quote attributed to celebrity goes viral online",
            "Outdated photo shared as recent event to mislead public",
            "False claims about election fraud spread without evidence",
            "Hoax article about product recall causes unnecessary panic",
            "Deepfake video circulates claiming to show fake incident",
            "Scam website promotes fake investment opportunity with lies"
        ],
        'label': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
                  0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
    }

    df = pd.DataFrame(training_data)
    return df

print("🚀 Starting Fake News Detection System...")
print("="*60)

# Create and preprocess dataset
print("\n📊 Creating training dataset...")
df = create_training_dataset()
print(f"✅ Created dataset with {len(df)} articles (15 Real, 15 Fake)")

# Preprocess data
print("\n🔧 Preprocessing text data...")
preprocessor = DataPreprocessor()
df_processed = preprocessor.preprocess_dataset(df)
print("✅ Preprocessing complete")

# Initialize detector
print("\n🤖 Initializing ML models...")
detector = FakeNewsDetector()

# Extract features
print("\n📈 Extracting TF-IDF features...")
X = detector.extract_features(df_processed['processed_text'])
y = df_processed['label']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"✅ Training samples: {X_train.shape[0]}, Testing samples: {X_test.shape[0]}")

# Train models
print("\n🎯 Training ML models...")
detector.train_models(X_train, y_train)

# Evaluate models
print("\n📊 Evaluating model performance...")
results = detector.evaluate_models(X_test, y_test)

# Display results
print("\n" + "="*60)
print("📊 MODEL EVALUATION RESULTS")
print("="*60)
results_df = pd.DataFrame(results).T
print(results_df.round(4))

print(f"\n🏆 Best Performing Model: {detector.best_model_name}")

# Generate confusion matrix for best model
y_pred = detector.best_model.predict(X_test)
viz = VisualizationModule()

print("\n🎨 Generating visualizations...")
viz.plot_model_comparison(results)
viz.plot_confusion_matrix(y_test, y_pred)

# Show best model metrics
best_metrics = results[detector.best_model_name]
print("\n" + "="*60)
print(f"📈 {detector.best_model_name} - Final Performance Metrics")
print("="*60)
print(f"Accuracy:  {best_metrics['Accuracy']:.4f}")
print(f"Precision: {best_metrics['Precision']:.4f}")
print(f"Recall:    {best_metrics['Recall']:.4f}")
print(f"F1 Score:  {best_metrics['F1 Score']:.4f}")

# Initialize storage and UI
print("\n🎨 Launching Interactive User Interface...")
storage = DataStorage()
ui = FakeNewsUI(detector, preprocessor, storage)
ui.create_ui()

print("\n✅ System Ready! Use the interface above to detect fake news.")